In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU    :', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU — Runtime > Change runtime type > T4 GPU')
!pip install pytorch-metric-learning -q
print('Done.')

PyTorch: 2.10.0+cu128
CUDA   : True
GPU    : Tesla T4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.8/127.8 kB 5.1 MB/s eta 0:00:00
Done.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

KAGGLE_USERNAME = 'your_username_here'
KAGGLE_KEY      = 'your_key_here'

os.environ['KAGGLE_USERNAME'] = KAGGLE_USERNAME
os.environ['KAGGLE_KEY']      = KAGGLE_KEY

os.makedirs('/content/data', exist_ok=True)
os.chdir('/content/data')
!pip install kaggle -q
!kaggle datasets download -d majdouline20/shapenetpart-dataset
!unzip -q shapenetpart-dataset.zip
os.chdir('/content')

DATA_ROOT = None
for root, dirs, files in os.walk('/content/data'):
    if '03001627' in dirs:
        DATA_ROOT = root
        break
print('Dataset:', DATA_ROOT)

Mounted at /content/drive
Dataset URL: https://www.kaggle.com/datasets/majdouline20/shapenetpart-dataset
License(s): MIT
100% 1.02G/1.02G [00:16<00:00, 66.5MB/s]

Dataset: /content/data/PartAnnotation


In [ ]:
import os
import copy
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from sklearn.metrics import jaccard_score, fbeta_score, precision_score, recall_score
from pytorch_metric_learning.losses import NTXentLoss
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Device: cuda


In [ ]:
CFG = {
    # Data
    'data_root'      : DATA_ROOT,
    'obj_class'      : 4,           # chair
    'num_points'     : 1024,
    'train_ratio'    : 0.8,
    # Training
    'batch_size'     : 8,
    'n_epochs'       : 50,          # more epochs — EMA prevents degradation
    'lr'             : 3e-4,
    'step_size'      : 20,
    'gamma'          : 0.5,
    # Model
    'n_fg'           : 256,
    'n_bg'           : 256,
    'group_size'     : 7,
    'emb_dim'        : 512,
    'dgcnn_k'        : 20,
    'num_parts'      : 2,
    # Loss weights — best values from previous experiments
    'repulsion'      : 0.5,
    'alpha'          : 1.0,
    'n_pos_pairs'    : 300,
    'n_neg_pairs'    : 600,
    'temp_point'     : 0.5,
    'temp_obj'       : 0.07,
    'lambda_spatial' : 0.01,        # best from ablation
    'k_spatial'      : 10,
    'lambda_entropy' : 0.0001,      # best from ablation
    # EMA Teacher-Student
    'ema_alpha'      : 0.999,       # teacher update momentum
    'lambda_consist' : 0.1,         # consistency loss weight — try 0.01, 0.1, 1.0
}

CLASS_TO_NAME = {
    0:'airplane',1:'bag',2:'cap',3:'car',4:'chair',5:'earphone',
    6:'guitar',7:'knife',8:'lamp',9:'laptop',10:'motorbike',
    11:'mug',12:'pistol',13:'rocket',14:'skateboard',15:'table'
}
print(f'Object: {CLASS_TO_NAME[CFG["obj_class"]]}')
print(f'Epochs: {CFG["n_epochs"]} | EMA α={CFG["ema_alpha"]} | λ_consist={CFG["lambda_consist"]}')

Object: chair
Epochs: 50 | EMA α=0.999 | λ_consist=0.1


In [ ]:
SYNSET_TO_CLASS = {
    '02691156':0,'02773838':1,'02954340':2,'02958343':3,
    '03001627':4,'03261776':5,'03467517':6,'03624134':7,
    '03636649':8,'03642806':9,'03790512':10,'03797390':11,
    '03948459':12,'04099429':13,'04225987':14,'04379243':15,
}

def load_pts(path):
    pts = []
    with open(path) as f:
        for line in f:
            v = line.strip().split()
            if len(v) >= 3: pts.append([float(x) for x in v[:3]])
    return np.array(pts, dtype='float32')

def load_seg(path):
    with open(path) as f:
        return np.array([int(l.strip()) for l in f if l.strip()], dtype='int64')

class ShapeNetPart(Dataset):
    def __init__(self, data_root, obj_class=4, partition='train',
                 num_points=1024, train_ratio=0.8):
        self.num_points = num_points
        synset  = next(s for s,c in SYNSET_TO_CLASS.items() if c==obj_class)
        syn_dir = os.path.join(data_root, synset)
        pts_dir = os.path.join(syn_dir,'points')
        seg_dir = os.path.join(syn_dir,'points_label')
        seg_map = {}
        if os.path.isdir(seg_dir):
            for entry in os.listdir(seg_dir):
                ep = os.path.join(seg_dir,entry)
                if entry.endswith('.seg'): seg_map[entry[:-4]] = ep
                elif os.path.isdir(ep):
                    for f in os.listdir(ep):
                        if f.endswith('.seg') and f[:-4] not in seg_map:
                            seg_map[f[:-4]] = os.path.join(ep,f)
        all_ids = sorted([f[:-4] for f in os.listdir(pts_dir) if f.endswith('.pts')])
        valid   = [i for i in all_ids if i in seg_map]
        np.random.seed(42)
        perm  = np.random.permutation(len(valid))
        split = int(len(valid)*train_ratio)
        chosen = [valid[i] for i in (perm[:split] if partition=='train' else perm[split:])]
        self.samples = [(os.path.join(pts_dir,s+'.pts'),seg_map[s]) for s in chosen]
        print(f'[{CLASS_TO_NAME[obj_class]}|{partition}] {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        pc  = load_pts(self.samples[idx][0])
        seg = load_seg(self.samples[idx][1])
        n   = min(len(pc),len(seg))
        pc,seg = pc[:n],seg[:n]
        idx_s = (np.random.choice(n,self.num_points,replace=False)
                 if n>=self.num_points
                 else np.random.choice(n,self.num_points,replace=True))
        pc,seg = pc[idx_s],seg[idx_s]
        pc -= pc.mean(0)
        s = np.max(np.linalg.norm(pc,axis=1))
        if s > 0: pc /= s
        return pc.astype('float32'), (seg>seg.min()).astype('int64')

train_ds = ShapeNetPart(CFG['data_root'],CFG['obj_class'],'train',
                        CFG['num_points'],CFG['train_ratio'])
test_ds  = ShapeNetPart(CFG['data_root'],CFG['obj_class'],'test',
                        CFG['num_points'],CFG['train_ratio'])
bs = min(CFG['batch_size'],len(train_ds))
train_loader = DataLoader(train_ds,batch_size=bs,shuffle=True,
                          drop_last=True,num_workers=2)
test_loader  = DataLoader(test_ds, batch_size=1,shuffle=False,
                          drop_last=False,num_workers=2)
print(f'Train batches: {len(train_loader)} | Test: {len(test_ds)}')

[chair|train] 5422 samples
[chair|test] 1356 samples
Train batches: 677 | Test: 1356


In [ ]:
# ── DGCNN ─────────────────────────────────────────────────────────────────────
def knn_graph(x,k):
    xt=x.permute(0,2,1); dist=torch.cdist(xt,xt)
    return dist.topk(k+1,dim=-1,largest=False).indices[:,:,1:]

def get_graph_feature(x,k=20):
    B,D,N=x.shape; idx=knn_graph(x,k)
    base=torch.arange(B,device=x.device).view(-1,1,1)*N
    flat=(idx+base).view(-1); xt=x.permute(0,2,1).contiguous()
    nbr=xt.view(B*N,-1)[flat].view(B,N,k,D); ctr=xt.view(B,N,1,D).expand(B,N,k,D)
    return torch.cat([nbr-ctr,ctr],3).permute(0,3,1,2).contiguous()

class DGCNN(nn.Module):
    def __init__(self,k=20,emb_dim=512,num_parts=2):
        super().__init__(); self.k=k
        self.c1=nn.Sequential(nn.Conv2d(6,  64,1,bias=False),nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.c2=nn.Sequential(nn.Conv2d(128,64,1,bias=False),nn.BatchNorm2d(64), nn.LeakyReLU(0.2))
        self.c3=nn.Sequential(nn.Conv2d(128,128,1,bias=False),nn.BatchNorm2d(128),nn.LeakyReLU(0.2))
        self.c4=nn.Sequential(nn.Conv2d(256,256,1,bias=False),nn.BatchNorm2d(256),nn.LeakyReLU(0.2))
        self.c5=nn.Sequential(nn.Conv1d(512,emb_dim,1,bias=False),
                               nn.BatchNorm1d(emb_dim),nn.LeakyReLU(0.2))
        self.head=nn.Sequential(
            nn.Linear(emb_dim*2,512,bias=False),nn.BatchNorm1d(512),
            nn.LeakyReLU(0.2),nn.Dropout(0.4),
            nn.Linear(512,256,bias=False),nn.BatchNorm1d(256),
            nn.LeakyReLU(0.2),nn.Dropout(0.4))
        self.entropy_head=nn.Conv1d(emb_dim,num_parts,1,bias=True)

    def forward(self,x):
        """
        x: (B,3,N)
        returns:
          point_feat  : (B, emb_dim, N)
          obj_feat    : (B, 256)
          part_probs  : (B, num_parts, N)  softmax probabilities
        """
        x1=self.c1(get_graph_feature(x,self.k)).max(-1)[0]
        x2=self.c2(get_graph_feature(x1,self.k)).max(-1)[0]
        x3=self.c3(get_graph_feature(x2,self.k)).max(-1)[0]
        x4=self.c4(get_graph_feature(x3,self.k)).max(-1)[0]
        pf=self.c5(torch.cat([x1,x2,x3,x4],1))             # (B,emb_dim,N)
        g=torch.cat([F.adaptive_max_pool1d(pf,1).squeeze(-1),
                      F.adaptive_avg_pool1d(pf,1).squeeze(-1)],1)
        part_probs=torch.softmax(self.entropy_head(pf),1)   # (B,num_parts,N)
        return pf, self.head(g), part_probs

# ── SAMPLER ───────────────────────────────────────────────────────────────────
class PointSampler(nn.Module):
    def __init__(self,num_out=256,bottleneck=128,group_size=7):
        super().__init__(); self.num_out=num_out; self.group_size=group_size
        self.enc=nn.Sequential(
            nn.Conv1d(3,64,1),nn.BatchNorm1d(64),nn.ReLU(),
            nn.Conv1d(64,128,1),nn.BatchNorm1d(128),nn.ReLU(),
            nn.Conv1d(128,bottleneck,1),nn.BatchNorm1d(bottleneck),nn.ReLU())
        self.dec=nn.Sequential(
            nn.Linear(bottleneck,256),nn.BatchNorm1d(256),nn.ReLU(),
            nn.Linear(256,512),nn.BatchNorm1d(512),nn.ReLU(),
            nn.Linear(512,num_out*3))
        self.sigma=nn.Parameter(torch.tensor(1.0)); self._pj=None

    def forward(self,x):
        B,N,_=x.shape; h=self.enc(x.permute(0,2,1)).max(-1)[0]
        simp=self.dec(h).view(B,self.num_out,3)
        dist=torch.cdist(simp,x); g=min(self.group_size,N)
        kd=dist.topk(g,dim=-1,largest=False).values
        ki=dist.topk(g,dim=-1,largest=False).indices
        ki_e=ki.unsqueeze(-1).expand(B,self.num_out,g,3)
        nbr=torch.gather(x.unsqueeze(1).expand(B,self.num_out,N,3),2,ki_e)
        w=torch.softmax(-kd/self.sigma.clamp(min=1e-2),-1)
        proj=(w.unsqueeze(-1)*nbr).sum(2)
        self._pj=((simp-proj)**2).sum(-1).mean()
        return simp,proj

    def get_simplification_loss(self,pc,simp,n,gamma=1.0):
        return torch.cdist(simp,pc).min(-1)[0].mean()

    def get_projection_loss(self):
        return self._pj if self._pj is not None else torch.tensor(0.0)

class CoSegNet(nn.Module):
    def __init__(self,n_fg=256,n_bg=256,group_size=7):
        super().__init__()
        self.fg=PointSampler(n_fg,group_size=group_size)
        self.bg=PointSampler(n_bg,group_size=group_size)

print('DGCNN + CoSegNet defined.')

DGCNN + CoSegNet defined.


In [ ]:
def chamfer_distance(pc1,pc2):
    d=torch.cdist(pc1,pc2); return d.min(2)[0],d.min(1)[0]

def spatial_consistency_loss(pf,xyz,k=10):
    """mean ||F_i - F_j||^2 for j in KNN_k(i)"""
    B,D,N=pf.shape; dist=torch.cdist(xyz,xyz)
    idx=dist.topk(k+1,dim=-1,largest=False).indices[:,:,1:]
    feats=pf.permute(0,2,1).contiguous(); flat=idx.reshape(B,-1)
    flat_e=flat.unsqueeze(-1).expand(B,N*k,D)
    nbr_f=torch.gather(feats,1,flat_e).view(B,N,k,D)
    fi=feats.unsqueeze(2).expand(B,N,k,D)
    return ((fi-nbr_f)**2).sum(-1).mean()

def entropy_loss(part_probs):
    """mean entropy — encourages confident per-point assignments"""
    p=part_probs.clamp(min=1e-8)
    return -(p*torch.log(p)).sum(dim=1).mean()

def consistency_loss(student_probs, teacher_probs):
    """
    EMA TEACHER-STUDENT CONSISTENCY LOSS
    ──────────────────────────────────────
    Forces student predictions to match teacher predictions.
    Teacher is detached — no gradients flow through teacher.

    student_probs : (B, num_parts, N)  from student
    teacher_probs : (B, num_parts, N)  from teacher  (detached)
    returns       : scalar loss
    """
    return ((student_probs - teacher_probs.detach()) ** 2).mean()

print('All loss functions defined:')
print('  chamfer_distance')
print('  spatial_consistency_loss')
print('  entropy_loss')
print('  consistency_loss  <-- NEW: EMA teacher-student')

All loss functions defined:
  chamfer_distance
  spatial_consistency_loss
  entropy_loss
  consistency_loss  <-- NEW: EMA teacher-student


In [ ]:
def update_ema(student_model, teacher_model, alpha=0.999):
    """
    EMA TEACHER UPDATE
    ───────────────────
    Called after every training batch.
    Teacher weights = alpha * teacher + (1-alpha) * student

    alpha=0.999 means teacher changes very slowly:
      - 99.9% of the time it keeps its own weights
      - 0.1% of the time it absorbs student weights
    This preserves the good structure from early training.

    IMPORTANT: Teacher parameters are NEVER updated by backprop.
               Only this function updates the teacher.
    """
    with torch.no_grad():
        for s_param, t_param in zip(student_model.parameters(),
                                     teacher_model.parameters()):
            t_param.data = alpha * t_param.data + (1.0 - alpha) * s_param.data


def create_teacher(student_model):
    """
    Create teacher as exact copy of student.
    Teacher has requires_grad=False — never updated by backprop.
    """
    teacher = copy.deepcopy(student_model)
    for param in teacher.parameters():
        param.requires_grad_(False)
    return teacher


def fps(pc,n):
    B,N,_=pc.shape
    if n>=N: return pc
    dev=pc.device; cents=torch.zeros(B,n,dtype=torch.long,device=dev)
    dist=torch.full((B,N),1e10,device=dev); far=torch.randint(0,N,(B,),device=dev)
    for i in range(n):
        cents[:,i]=far
        c=pc[torch.arange(B,device=dev),far].unsqueeze(1)
        d=((pc-c)**2).sum(-1); dist=torch.min(dist,d); far=dist.argmax(-1)
    return torch.gather(pc,1,cents.unsqueeze(-1).expand(B,n,3))

print('EMA functions defined.')
print(f'Will use alpha={CFG["ema_alpha"]} for teacher updates.')

EMA functions defined.
Will use alpha=0.999 for teacher updates.


In [ ]:
def run_training_ema(
    tag              = 'ema_model',
    lambda_spatial   = 0.01,
    lambda_entropy   = 0.0001,
    lambda_consist   = 0.1,
    ema_alpha        = 0.999,
    k_spatial        = 10,
    use_ema          = True,       # set False to run without EMA for comparison
):
    """
    Full training loop with EMA Teacher-Student framework.

    When use_ema=True:
      L = L_contrastive + λ_sp*L_spatial + λ_ent*L_entropy + λ_c*L_consistency
      Teacher updated every batch via EMA.

    When use_ema=False:
      L = L_contrastive + λ_sp*L_spatial + λ_ent*L_entropy
      (baseline without EMA, for comparison)
    """
    print(f'\n{"="*60}')
    print(f'  {tag.upper()}')
    print(f'  EMA={use_ema} | α={ema_alpha} | λ_consist={lambda_consist}')
    print(f'  λ_spatial={lambda_spatial} | λ_entropy={lambda_entropy} | k={k_spatial}')
    print(f'  epochs={CFG["n_epochs"]}')
    print(f'{"="*60}\n')

    save_dir = f'/content/results/{tag}'
    os.makedirs(save_dir, exist_ok=True)
    log_path = f'{save_dir}/log.txt'
    open(log_path,'w').close()

    def log(msg):
        print(msg)
        with open(log_path,'a') as f: f.write(msg+'\n')

    # ── Student models ────────────────────────────────────────────────────────
    student_net  = CoSegNet(CFG['n_fg'],CFG['n_bg'],CFG['group_size']).to(DEVICE)
    student_feat = nn.DataParallel(
        DGCNN(k=CFG['dgcnn_k'],emb_dim=CFG['emb_dim'],num_parts=CFG['num_parts'])
    ).to(DEVICE)

    # ── Teacher models (EMA copies, no gradients) ─────────────────────────────
    if use_ema:
        teacher_net  = create_teacher(student_net)
        teacher_feat = create_teacher(student_feat)
        log('Teacher models created (EMA, no backprop).')

    # ── Optimiser — only student parameters ──────────────────────────────────
    params = list(student_net.parameters()) + list(student_feat.parameters())
    opt    = optim.AdamW(params, lr=CFG['lr'], weight_decay=1e-4)
    sched  = StepLR(opt, step_size=CFG['step_size'], gamma=CFG['gamma'])

    pt_fn  = NTXentLoss(temperature=CFG['temp_point'])
    obj_fn = NTXentLoss(temperature=CFG['temp_obj'])

    history  = {
        'iou':[], 'f1':[], 'loss':[],
        'loss_contrastive':[], 'loss_spatial':[],
        'loss_entropy':[], 'loss_consistency':[]
    }
    best_f1  = 0.0
    best_iou = 0.0

    for epoch in range(CFG['n_epochs']):
        student_net.train()
        student_feat.train()
        if use_ema:
            # Teacher always in eval mode — it never does backprop
            teacher_net.eval()
            teacher_feat.eval()

        running        = 0.0
        run_contrast   = 0.0
        run_spatial    = 0.0
        run_entropy    = 0.0
        run_consist    = 0.0
        nb             = 0

        for coord, label in train_loader:
            coord = fps(coord.to(DEVICE), CFG['num_points'])

            # ── Student forward pass ──────────────────────────────────────────
            fg_simp, fg_coord = student_net.fg(coord)
            bg_simp, bg_coord = student_net.bg(coord)

            d1,d2    = chamfer_distance(fg_coord, bg_coord)
            rep_loss = (torch.clamp(CFG['repulsion']-d1,min=0).mean() +
                        torch.clamp(CFG['repulsion']-d2,min=0).mean())

            fg_pf, fg_obj, fg_probs_student = student_feat(
                fg_coord.permute(0,2,1).contiguous()
            )
            bg_pf, bg_obj, bg_probs_student = student_feat(
                bg_coord.permute(0,2,1).contiguous()
            )

            samp = CFG['alpha'] * (
                student_net.fg.get_simplification_loss(coord,fg_simp,CFG['n_fg']) +
                student_net.fg.get_projection_loss() +
                student_net.bg.get_simplification_loss(coord,bg_simp,CFG['n_bg']) +
                student_net.bg.get_projection_loss()
            )

            # Point contrastive loss
            B = coord.shape[0]
            ptl = []
            for i in range(B):
                of  = fg_pf[i].permute(1,0)
                bf  = bg_pf[i].permute(1,0)
                emb = torch.cat([of,bf],0)
                lbl = torch.cat([torch.zeros(of.shape[0]),
                                  torch.arange(1,bf.shape[0]+1)]).long().to(DEVICE)
                pos = torch.randint(0,of.shape[0],(CFG['n_pos_pairs'],2),device=DEVICE)
                neg = torch.randint(0,bf.shape[0],(CFG['n_neg_pairs'],2),device=DEVICE)
                neg[:,1] += of.shape[0]
                ptl.append(pt_fn(emb,lbl,(pos[:,0],pos[:,1],neg[:,0],neg[:,1])))
            pt_loss = torch.stack(ptl).mean()

            emb = torch.cat([fg_obj,bg_obj],0)
            lbl = torch.cat([torch.zeros(fg_obj.shape[0]),
                              torch.arange(1,bg_obj.shape[0]+1)]).long().to(DEVICE)
            obj_loss = obj_fn(emb,lbl)

            contrastive_loss = pt_loss + obj_loss + samp + rep_loss

            # Spatial loss
            sp_loss = spatial_consistency_loss(fg_pf, fg_coord, k=k_spatial)

            # Entropy loss
            ent_loss = (entropy_loss(fg_probs_student) +
                        entropy_loss(bg_probs_student)) / 2

            # ── Teacher forward pass + Consistency loss ───────────────────────
            if use_ema:
                with torch.no_grad():
                    # Teacher processes same points — no grad
                    _, _, fg_probs_teacher = teacher_feat(
                        fg_coord.permute(0,2,1).contiguous()
                    )
                    _, _, bg_probs_teacher = teacher_feat(
                        bg_coord.permute(0,2,1).contiguous()
                    )

                consist_loss = (
                    consistency_loss(fg_probs_student, fg_probs_teacher) +
                    consistency_loss(bg_probs_student, bg_probs_teacher)
                ) / 2
            else:
                consist_loss = torch.tensor(0.0, device=DEVICE)

            # ── Total loss ────────────────────────────────────────────────────
            loss = (contrastive_loss
                    + lambda_spatial * sp_loss
                    + lambda_entropy * ent_loss
                    + lambda_consist * consist_loss)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, max_norm=1.0)
            opt.step()

            # ── EMA teacher update (after every batch) ────────────────────────
            if use_ema:
                update_ema(student_net,  teacher_net,  alpha=ema_alpha)
                update_ema(student_feat, teacher_feat, alpha=ema_alpha)

            running      += loss.item()
            run_contrast += contrastive_loss.item()
            run_spatial  += sp_loss.item()
            run_entropy  += ent_loss.item()
            run_consist  += consist_loss.item()
            nb += 1

        sched.step()

        # ── Validation using TEACHER (more stable than student) ───────────────
        eval_net  = teacher_net  if use_ema else student_net
        eval_feat = teacher_feat if use_ema else student_feat
        eval_net.eval()
        eval_feat.eval()

        preds, trues = [], []
        with torch.no_grad():
            for coord, label in test_loader:
                coord  = fps(coord.to(DEVICE), CFG['num_points'])
                _,fg_c = eval_net.fg(coord)
                dist   = torch.cdist(coord,fg_c).squeeze(0)
                min_d  = dist.min(-1)[0]
                thresh = min_d.topk(CFG['n_fg'],largest=False).values[-1]
                preds.append((min_d<=thresh).long().cpu().numpy())
                trues.append(label.squeeze().numpy())

        yp = np.concatenate(preds)
        yt = np.concatenate(trues)
        iou = jaccard_score(yt,yp,zero_division=0)
        f1  = fbeta_score(yt,yp,beta=0.75,zero_division=0)
        pre = precision_score(yt,yp,zero_division=0)
        rec = recall_score(yt,yp,zero_division=0)

        history['iou'].append(iou)
        history['f1'].append(f1)
        history['loss'].append(running/nb)
        history['loss_contrastive'].append(run_contrast/nb)
        history['loss_spatial'].append(run_spatial/nb)
        history['loss_entropy'].append(run_entropy/nb)
        history['loss_consistency'].append(run_consist/nb)

        consist_str = f' | L_consist {run_consist/nb:.4f}' if use_ema else ''
        log(f'[{tag}] E{epoch:03d} | loss {running/nb:.4f} | '
            f'IOU {iou:.4f} | F1 {f1:.4f} | '
            f'P {pre:.3f} | R {rec:.3f}'
            + consist_str)

        if f1 > best_f1:
            best_f1=f1; best_iou=iou
            torch.save({
                'student_net' : student_net.state_dict(),
                'student_feat': student_feat.state_dict(),
                'teacher_net' : teacher_net.state_dict()  if use_ema else None,
                'teacher_feat': teacher_feat.state_dict() if use_ema else None,
            }, f'{save_dir}/model_best.pt')
            log(f'  --> Best: IOU={best_iou:.4f}  F1={best_f1:.4f}')

    log(f'\nFINISHED [{tag}] | Best IOU={best_iou:.4f} | Best F1={best_f1:.4f}')
    return history, best_iou, best_f1

print('Training function ready.')

Training function ready.


In [ ]:
os.makedirs('/content/results', exist_ok=True)

# ── Run 1: Without EMA (best previous config, 50 epochs) ─────────────────────
h_no_ema, iou_no_ema, f1_no_ema = run_training_ema(
    tag='no_ema_50ep',
    lambda_spatial=0.01,
    lambda_entropy=0.0001,
    lambda_consist=0.0,    # no consistency loss
    use_ema=False,
)
print(f'\nWithout EMA: IOU={iou_no_ema:.4f}  F1={f1_no_ema:.4f}')


  NO_EMA_50EP
  EMA=False | α=0.999 | λ_consist=0.0
  λ_spatial=0.01 | λ_entropy=0.0001 | k=10
  epochs=50

[no_ema_50ep] E000 | loss 6.1329 | IOU 0.1464 | F1 0.2637 | P 0.288 | R 0.230
  --> Best: IOU=0.1464  F1=0.2637
[no_ema_50ep] E001 | loss 4.2996 | IOU 0.1463 | F1 0.2635 | P 0.287 | R 0.230
[no_ema_50ep] E002 | loss 3.3896 | IOU 0.1273 | F1 0.2331 | P 0.254 | R 0.203
[no_ema_50ep] E003 | loss 2.8585 | IOU 0.1166 | F1 0.2155 | P 0.235 | R 0.188
[no_ema_50ep] E004 | loss 2.5134 | IOU 0.1140 | F1 0.2112 | P 0.230 | R 0.184
[no_ema_50ep] E005 | loss 2.3392 | IOU 0.0921 | F1 0.1741 | P 0.190 | R 0.152
[no_ema_50ep] E006 | loss 2.2258 | IOU 0.1022 | F1 0.1914 | P 0.209 | R 0.167
[no_ema_50ep] E007 | loss 2.1409 | IOU 0.0878 | F1 0.1666 | P 0.182 | R 0.145
[no_ema_50ep] E008 | loss 1.9855 | IOU 0.0554 | F1 0.1084 | P 0.118 | R 0.094
[no_ema_50ep] E009 | loss 1.5750 | IOU 0.0675 | F1 0.1306 | P 0.142 | R 0.114
[no_ema_50ep] E010 | loss 1.3384 | IOU 0.0604 | F1 0.1176 | P 0.128 | R 0.102

In [ ]:
# ── Run 2: With EMA Teacher-Student ──────────────────────────────────────────
h_ema, iou_ema, f1_ema = run_training_ema(
    tag='ema_model',
    lambda_spatial=0.01,
    lambda_entropy=0.0001,
    lambda_consist=0.1,    # consistency loss weight
    ema_alpha=0.999,
    use_ema=True,
)
print(f'\nWith EMA: IOU={iou_ema:.4f}  F1={f1_ema:.4f}')


  EMA_MODEL
  EMA=True | α=0.999 | λ_consist=0.1
  λ_spatial=0.01 | λ_entropy=0.0001 | k=10
  epochs=50

Teacher models created (EMA, no backprop).
[ema_model] E000 | loss 6.0301 | IOU 0.1003 | F1 0.1883 | P 0.205 | R 0.164 | L_consist 0.0007
  --> Best: IOU=0.1003  F1=0.1883
[ema_model] E001 | loss 4.0421 | IOU 0.1041 | F1 0.1946 | P 0.212 | R 0.170 | L_consist 0.0001
  --> Best: IOU=0.1041  F1=0.1946
[ema_model] E002 | loss 3.1942 | IOU 0.1040 | F1 0.1944 | P 0.212 | R 0.169 | L_consist 0.0001
[ema_model] E003 | loss 2.8349 | IOU 0.1014 | F1 0.1901 | P 0.207 | R 0.166 | L_consist 0.0001
[ema_model] E004 | loss 2.6193 | IOU 0.1020 | F1 0.1911 | P 0.208 | R 0.167 | L_consist 0.0000
[ema_model] E005 | loss 2.3934 | IOU 0.1014 | F1 0.1900 | P 0.207 | R 0.166 | L_consist 0.0001
[ema_model] E006 | loss 2.2869 | IOU 0.1005 | F1 0.1884 | P 0.205 | R 0.164 | L_consist 0.0001
[ema_model] E007 | loss 2.1745 | IOU 0.0999 | F1 0.1876 | P 0.205 | R 0.163 | L_consist 0.0000
[ema_model] E008 | loss

In [ ]:
epochs = range(1, CFG['n_epochs']+1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('EMA Teacher-Student vs No EMA\nShapeNet Part — Chair',
             fontsize=14, fontweight='bold')

# IOU curve
ax = axes[0,0]
ax.plot(epochs, h_no_ema['iou'], label='Without EMA', color='steelblue', linewidth=2)
ax.plot(epochs, h_ema['iou'],    label='With EMA (α=0.999)', color='darkorange', linewidth=2)
ax.axhline(max(h_no_ema['iou']), color='steelblue', linestyle='--', alpha=0.5,
           label=f'No EMA best ({max(h_no_ema["iou"]):.4f})')
ax.axhline(max(h_ema['iou']),    color='darkorange', linestyle='--', alpha=0.5,
           label=f'EMA best ({max(h_ema["iou"]):.4f})')
ax.set_title('IOU over Epochs', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('IOU')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# F1 curve
ax = axes[0,1]
ax.plot(epochs, h_no_ema['f1'], label='Without EMA', color='steelblue', linewidth=2)
ax.plot(epochs, h_ema['f1'],    label='With EMA', color='darkorange', linewidth=2)
ax.set_title('F1 Score over Epochs', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('F1')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Total loss
ax = axes[1,0]
ax.plot(epochs, h_no_ema['loss'], label='Without EMA', color='steelblue', linewidth=2)
ax.plot(epochs, h_ema['loss'],    label='With EMA', color='darkorange', linewidth=2)
ax.set_title('Total Loss over Epochs', fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Consistency loss (EMA only)
ax = axes[1,1]
ax.plot(epochs, h_ema['loss_consistency'],
        label='Consistency loss (EMA)', color='green', linewidth=2)
ax.set_title('Consistency Loss over Epochs\n(How close student is to teacher)',
             fontsize=12)
ax.set_xlabel('Epoch'); ax.set_ylabel('L_consistency')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.annotate('Should decrease over time\n(student converging to teacher)',
            xy=(0.5, 0.7), xycoords='axes fraction',
            fontsize=9, color='gray', ha='center')

plt.tight_layout()
plt.savefig('/content/results/ema_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ema_comparison.png')

print('\n' + '='*55)
print(f'{"Config":<30} {"Best IOU":>12} {"Best F1":>12}')
print('='*55)
print(f'{"Without EMA (50 epochs)":<30} {iou_no_ema:>12.4f} {f1_no_ema:>12.4f}')
print(f'{"With EMA (50 epochs)":<30} {iou_ema:>12.4f} {f1_ema:>12.4f}')
print('='*55)
print(f'IOU change from EMA: {iou_ema - iou_no_ema:+.4f}')
print(f'F1  change from EMA: {f1_ema  - f1_no_ema:+.4f}')

In [ ]:
# Compute IOU variance in early vs late epochs
# to quantify stability improvement

n = len(h_no_ema['iou'])
early = slice(0, n//2)    # first half of training
late  = slice(n//2, n)    # second half of training

no_ema_early_var = np.var(h_no_ema['iou'][early.start:early.stop])
no_ema_late_var  = np.var(h_no_ema['iou'][late.start:late.stop])
ema_early_var    = np.var(h_ema['iou'][early.start:early.stop])
ema_late_var     = np.var(h_ema['iou'][late.start:late.stop])

no_ema_early_mean = np.mean(h_no_ema['iou'][early.start:early.stop])
no_ema_late_mean  = np.mean(h_no_ema['iou'][late.start:late.stop])
ema_early_mean    = np.mean(h_ema['iou'][early.start:early.stop])
ema_late_mean     = np.mean(h_ema['iou'][late.start:late.stop])

print('='*65)
print('  STABILITY ANALYSIS')
print('  (Lower variance = more stable training)')
print('='*65)
print(f'\n{"":<30} {"Early epochs":>15} {"Late epochs":>15}')
print(f'{"":<30} {"mean  /  var":>15} {"mean  /  var":>15}')
print('-'*65)
print(f'{"Without EMA":<30} '
      f'{no_ema_early_mean:.4f} / {no_ema_early_var:.6f}  '
      f'{no_ema_late_mean:.4f} / {no_ema_late_var:.6f}')
print(f'{"With EMA":<30} '
      f'{ema_early_mean:.4f} / {ema_early_var:.6f}  '
      f'{ema_late_mean:.4f} / {ema_late_var:.6f}')
print('='*65)

if ema_late_var < no_ema_late_var:
    reduction = 100*(1 - ema_late_var/no_ema_late_var)
    print(f'\nEMA reduced late-epoch variance by {reduction:.1f}%')
    print('EMA successfully stabilized training in later epochs.')
else:
    print('\nEMA did not reduce variance — try adjusting ema_alpha or lambda_consist.')

if ema_late_mean > no_ema_late_mean:
    print(f'EMA improved late-epoch mean IOU by {ema_late_mean-no_ema_late_mean:+.4f}')
    print('This confirms EMA prevents the degradation seen after early epochs.')
else:
    print(f'Late-epoch mean IOU change: {ema_late_mean-no_ema_late_mean:+.4f}')

In [ ]:
# Reduce epochs for this ablation to save time
original_epochs = CFG['n_epochs']
CFG['n_epochs']  = 20

alpha_results = {}
for alpha in [0.99, 0.999, 0.9999]:
    h, iou, f1 = run_training_ema(
        tag=f'ema_alpha_{str(alpha).replace(".","p")}',
        lambda_spatial=0.01,
        lambda_entropy=0.0001,
        lambda_consist=0.1,
        ema_alpha=alpha,
        use_ema=True,
    )
    alpha_results[alpha] = {'iou': iou, 'f1': f1, 'history': h}
    print(f'alpha={alpha} -> IOU={iou:.4f}  F1={f1:.4f}')

CFG['n_epochs'] = original_epochs  # restore

# Plot
epochs_short = range(1, 21)
fig, (ax1, ax2) = plt.subplots(1,2,figsize=(14,5))
fig.suptitle('EMA Alpha Ablation (λ_consist=0.1)', fontsize=13, fontweight='bold')

colors_alpha = ['steelblue','darkorange','green']
for (alpha, res), col in zip(alpha_results.items(), colors_alpha):
    ax1.plot(epochs_short, res['history']['iou'],
             label=f'α={alpha}', color=col, linewidth=2)
    ax2.plot(epochs_short, res['history']['loss_consistency'],
             label=f'α={alpha}', color=col, linewidth=2)

ax1.set_title('IOU vs Epoch'); ax1.set_xlabel('Epoch')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax2.set_title('Consistency Loss vs Epoch'); ax2.set_xlabel('Epoch')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/results/ema_alpha_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n' + '='*45)
print(f'{"EMA Alpha":>12} {"Best IOU":>12} {"Best F1":>12}')
print('='*45)
for alpha, res in alpha_results.items():
    print(f'{alpha:>12} {res["iou"]:>12.4f} {res["f1"]:>12.4f}')
print('='*45)

In [ ]:
def load_ema_model(tag, use_teacher=True):
    path = f'/content/results/{tag}/model_best.pt'
    if not os.path.exists(path):
        print(f'Not found: {path}'); return None, None
    ckpt = torch.load(path, map_location=DEVICE)
    net  = CoSegNet(CFG['n_fg'],CFG['n_bg'],CFG['group_size']).to(DEVICE)
    fext = nn.DataParallel(
        DGCNN(k=CFG['dgcnn_k'],emb_dim=CFG['emb_dim'],num_parts=CFG['num_parts'])
    ).to(DEVICE)
    key_net  = 'teacher_net'  if (use_teacher and ckpt.get('teacher_net') is not None) else 'student_net'
    key_feat = 'teacher_feat' if (use_teacher and ckpt.get('teacher_feat') is not None) else 'student_feat'
    net.load_state_dict(ckpt[key_net])
    fext.load_state_dict(ckpt[key_feat])
    net.eval(); fext.eval()
    print(f'  Loaded {tag} using {key_net}/{key_feat}')
    return net, fext

def predict(net, feat_net, pc_np):
    with torch.no_grad():
        coord = fps(torch.FloatTensor(pc_np).unsqueeze(0).to(DEVICE), CFG['num_points'])
        _, fg_c = net.fg(coord)
        dist    = torch.cdist(coord,fg_c).squeeze(0)
        min_d   = dist.min(-1)[0]
        thresh  = min_d.topk(CFG['n_fg'],largest=False).values[-1]
        pred    = (min_d<=thresh).long().cpu().numpy()
    return coord.squeeze(0).cpu().numpy(), pred

print('Loading models...')
net_no_ema, feat_no_ema = load_ema_model('no_ema_50ep', use_teacher=False)
net_ema,    feat_ema    = load_ema_model('ema_model',   use_teacher=True)

# Visualize 4 samples
n_show = 4
fig = plt.figure(figsize=(15, 5*n_show))
fig.suptitle('EMA Teacher vs No EMA — Segmentation Quality\n'
             'Blue=Background  |  Red=Foreground',
             fontsize=13, fontweight='bold')

for row in range(n_show):
    pc_np, true_lbl = test_ds[row*4]

    # Ground truth
    ax = fig.add_subplot(n_show,3,row*3+1,projection='3d')
    colors = np.where(true_lbl==1,'red','cornflowerblue')
    ax.scatter(pc_np[:,0],pc_np[:,1],pc_np[:,2],c=colors,s=5,alpha=0.85)
    ax.set_title(f'Sample {row+1}\nGround Truth',fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

    # Without EMA
    if net_no_ema:
        coord_b,pred_b = predict(net_no_ema,feat_no_ema,pc_np)
        n = min(len(true_lbl),len(pred_b))
        iou_b = jaccard_score(true_lbl[:n],pred_b[:n],zero_division=0)
        ax = fig.add_subplot(n_show,3,row*3+2,projection='3d')
        colors = np.where(pred_b==1,'red','cornflowerblue')
        ax.scatter(coord_b[:,0],coord_b[:,1],coord_b[:,2],c=colors,s=5,alpha=0.85)
        ax.set_title(f'Without EMA\nIOU={iou_b:.3f}',fontsize=9)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

    # With EMA teacher
    if net_ema:
        coord_e,pred_e = predict(net_ema,feat_ema,pc_np)
        n = min(len(true_lbl),len(pred_e))
        iou_e = jaccard_score(true_lbl[:n],pred_e[:n],zero_division=0)
        ax = fig.add_subplot(n_show,3,row*3+3,projection='3d')
        colors = np.where(pred_e==1,'red','cornflowerblue')
        ax.scatter(coord_e[:,0],coord_e[:,1],coord_e[:,2],c=colors,s=5,alpha=0.85)
        ax.set_title(f'With EMA Teacher\nIOU={iou_e:.3f}',fontsize=9)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])

plt.tight_layout()
plt.savefig('/content/results/ema_visualization.png',dpi=150,bbox_inches='tight')
plt.show()
print('Saved: ema_visualization.png')

In [ ]:
import shutil
dest = '/content/drive/MyDrive/BTP_coseg_EMA'
os.makedirs(dest, exist_ok=True)
shutil.copytree('/content/results', dest, dirs_exist_ok=True)
print('Saved to:', dest)
for root,dirs,files in os.walk(dest):
    for f in files:
        fp=os.path.join(root,f)
        print(f'  {fp.replace(dest,"")}  ({os.path.getsize(fp)//1024} KB)')